#### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#### Loading the Dataset

In [2]:
data_path = Path("../data/processed/online_retail_raw.parquet")

df = pd.read_parquet(data_path)

print("Dataset Shape:", df.shape)
print("\nDataset Columns:")
print(df.columns.tolist())

print("\nFirst Five Records:")
display(df.head())

Dataset Shape: (541909, 8)

Dataset Columns:
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

First Five Records:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


#### Inspecting Data Types and Missing Values

In [3]:
data_quality_summary = pd.DataFrame({
    "Data Type": df.dtypes.astype(str),
    "Non-Null Count": df.notna().sum(),
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2),
    "Unique Values": df.nunique(dropna=True)
})

display(data_quality_summary)

print("Total Rows:", len(df))
print("Duplicate Rows:", df.duplicated().sum())
print("Date Range:", df["InvoiceDate"].min(), "to", df["InvoiceDate"].max())

,Data Type,Non-Null Count,Missing Count,Missing Percentage,Unique Values
InvoiceNo,string,541909,0,0.00,25900
StockCode,string,541909,0,0.00,4070
Description,string,540455,1454,0.27,4223
Quantity,int64,541909,0,0.00,722
InvoiceDate,datetime64[us],541909,0,0.00,23260
UnitPrice,float64,541909,0,0.00,1630
CustomerID,float64,406829,135080,24.93,4372
Country,string,541909,0,0.00,38


Total Rows: 541909
Duplicate Rows: 5268
Date Range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


**Key Observations**
- **The dataset contains 541,909 transaction records across 8 columns**, covering December 1, 2010, through December 9, 2011. All essential revenue calculation fields (`InvoiceDate`, `Quantity`, and `UnitPrice`) are complete.
- **Missing values are concentrated in `CustomerID` (24.93%) and `Description` (0.27%)**, neither of which is required to calculate daily sales revenue.
- **The dataset contains 5,268 fully duplicated records**, requiring careful consideration before aggregation because removing them could change the calculated revenue totals.

**Data Preparation Decisions**
- **Retain records with missing `CustomerID` or `Description`**, as these fields are not necessary for the daily revenue forecasting target.
- **Preserve duplicate transaction records initially**, avoiding unsupported assumptions about whether identical invoice lines represent accidental duplication or legitimate transactions.
- **Use `InvoiceDate`, `Quantity`, and `UnitPrice` to construct the revenue target**, while applying the merchandise exclusions and cancellation treatment established during exploratory analysis.

#### Preparing Transaction-Level Revenue Data

In [4]:
transactions_df = df.copy()

transactions_df["Transaction_Date"] = (
    transactions_df["InvoiceDate"].dt.normalize()
)

transactions_df["Line_Amount"] = (
    transactions_df["Quantity"] * transactions_df["UnitPrice"]
)

transactions_df["Is_Cancellation"] = (
    transactions_df["InvoiceNo"].str.startswith("C")
)

print("Transaction-Level Dataset Shape:", transactions_df.shape)

print("\nTransaction Type Summary:")
display(
    transactions_df["Is_Cancellation"]
    .value_counts()
    .rename_axis("Is_Cancellation")
    .reset_index(name="Record_Count")
)

print("\nRevenue Contribution Summary:")
display(
    transactions_df.groupby("Is_Cancellation")["Line_Amount"]
    .agg(["count", "sum", "mean", "min", "max"])
    .round(2)
)

print("\nFirst Five Prepared Records:")
display(
    transactions_df[
        [
            "InvoiceNo",
            "Transaction_Date",
            "StockCode",
            "Quantity",
            "UnitPrice",
            "Line_Amount",
            "Is_Cancellation"
        ]
    ].head()
)

Transaction-Level Dataset Shape: (541909, 11)

Transaction Type Summary:


,Is_Cancellation,Record_Count
0,False,532621
1,True,9288



Revenue Contribution Summary:


,count,sum,mean,min,max
Is_Cancellation,,,,,
False,532621,10644560.42,19.99,-11062.06,168469.60
True,9288,-896812.49,-96.56,-168469.60,-0.12



First Five Prepared Records:


,InvoiceNo,Transaction_Date,StockCode,Quantity,UnitPrice,Line_Amount,Is_Cancellation
0,536365,2010-12-01,85123A,6,2.55,15.30,False
1,536365,2010-12-01,71053,6,3.39,20.34,False
2,536365,2010-12-01,84406B,8,2.75,22.00,False
3,536365,2010-12-01,84029G,6,3.39,20.34,False
4,536365,2010-12-01,84029E,6,3.39,20.34,False


**Key Observations**
- **The dataset contains 541,909 invoice-line records**, including 532,621 non-cancellation records (98.29%) and 9,288 cancellation records (1.71%).
- **Non-cancellation records contribute 10,644,560.42 in revenue, while cancellation records contribute -896,812.49**, demonstrating the substantial impact of cancellations on net revenue calculations.
- **Negative revenue is not limited to cancellation invoices**, as non-cancellation records also contain negative line amounts, with a minimum of -11,062.06.

**Data Preparation Decisions**
- **Retain cancellation records and negative line amounts** to preserve the net revenue definition established during exploratory analysis.
- **Avoid treating the cancellation flag as a complete indicator of negative revenue**, since negative amounts also occur outside explicitly identified cancellation invoices.
- **Apply the established non-merchandise exclusions before daily aggregation**, ensuring that the cleaned transaction dataset and forecasting target remain consistent with the EDA methodology.

#### Filtering Non-Merchandise Transactions

In [5]:
non_merchandise_codes = [
    "POST", "DOT", "C2", "M", "D",
    "S", "BANK CHARGES", "AMAZONFEE",
    "CRUK", "B"
]

cleaned_df = transactions_df.loc[
    ~transactions_df["StockCode"].isin(non_merchandise_codes)
].copy()

filtering_summary = pd.DataFrame({
    "Metric": [
        "Original Invoice Lines",
        "Excluded Non-Merchandise Lines",
        "Remaining Merchandise Lines",
        "Records Retained (%)",
        "Original Net Revenue",
        "Excluded Net Revenue",
        "Cleaned Net Merchandise Revenue",
        "Remaining Cancellation Lines",
        "Remaining Duplicate Lines"
    ],
    "Value": [
        len(transactions_df),
        len(transactions_df) - len(cleaned_df),
        len(cleaned_df),
        round(len(cleaned_df) / len(transactions_df) * 100, 2),
        round(transactions_df["Line_Amount"].sum(), 2),
        round(
            transactions_df.loc[
                transactions_df["StockCode"].isin(non_merchandise_codes),
                "Line_Amount"
            ].sum(), 2
        ),
        round(cleaned_df["Line_Amount"].sum(), 2),
        int(cleaned_df["Is_Cancellation"].sum()),
        int(cleaned_df.duplicated().sum())
    ]
})

print("Merchandise Filtering Summary:")
display(filtering_summary)

print("\nRemaining Missing Values:")
display(cleaned_df.isna().sum().to_frame("Missing_Count"))

print("\nCleaned Dataset Shape:", cleaned_df.shape)

Merchandise Filtering Summary:


,Metric,Value
0,Original Invoice Lines,541909.00
1,Excluded Non-Merchandise Lines,2911.00
2,Remaining Merchandise Lines,538998.00
3,Records Retained (%),99.46
4,Original Net Revenue,9747747.93
5,Excluded Net Revenue,-45649.31
6,Cleaned Net Merchandise Revenue,9793397.24
7,Remaining Cancellation Lines,8704.00
8,Remaining Duplicate Lines,5262.00



Remaining Missing Values:


,Missing_Count
InvoiceNo,0
StockCode,0
Description,1448
Quantity,0
InvoiceDate,0
UnitPrice,0
CustomerID,134085
Country,0
Transaction_Date,0
Line_Amount,0



Cleaned Dataset Shape: (538998, 11)


**Key Observations**
- **Non-merchandise filtering removes 2,911 invoice lines (0.54%)**, retaining 538,998 records and producing a net merchandise revenue total of 9,793,397.24.
- **The excluded records contribute -45,649.31 in net revenue**, resulting in a higher merchandise revenue total than the original unfiltered revenue of 9,747,747.93.
- **The filtered dataset retains 8,704 cancellation lines and 5,262 duplicate records**, while missing values remain limited to `CustomerID` (134,085) and `Description` (1,448).

**Data Preparation Decisions**
- **Retain the filtered transaction records as the cleaned merchandise dataset**, preserving cancellation activity, negative revenue contributions, and duplicate invoice lines for consistency with exploratory analysis.
- **Preserve missing customer identifiers and product descriptions**, since neither field is required for calculating daily merchandise revenue.
- **Save the cleaned transaction-level dataset in Parquet format** to support reproducible daily aggregation and reuse across subsequent notebooks.

#### Saving the Cleaned Transaction Dataset

In [6]:
cleaned_data_path = Path("../data/processed/online_retail_cleaned.parquet")

cleaned_data_path.parent.mkdir(parents=True, exist_ok=True)

cleaned_df.to_parquet(
    cleaned_data_path,
    index=False
)

saved_cleaned_df = pd.read_parquet(cleaned_data_path)

assert saved_cleaned_df.shape == cleaned_df.shape
assert np.isclose(
    saved_cleaned_df["Line_Amount"].sum(),
    cleaned_df["Line_Amount"].sum()
)

print("Cleaned Dataset Saved Successfully")
print("File Path:", cleaned_data_path)
print("Saved Dataset Shape:", saved_cleaned_df.shape)
print("Saved Net Merchandise Revenue:", round(saved_cleaned_df["Line_Amount"].sum(), 2))

Cleaned Dataset Saved Successfully
File Path: ..\data\processed\online_retail_cleaned.parquet
Saved Dataset Shape: (538998, 11)
Saved Net Merchandise Revenue: 9793397.24


#### Aggregating Daily Net Merchandise Revenue

In [7]:
daily_revenue_df = (
    cleaned_df.groupby("Transaction_Date", as_index=False)
    .agg(
        Daily_Revenue=("Line_Amount", "sum")
    )
    .sort_values("Transaction_Date")
    .reset_index(drop=True)
)

daily_revenue_df["Transaction_Date"] = pd.to_datetime(
    daily_revenue_df["Transaction_Date"]
)

print("Daily Revenue Dataset Shape:", daily_revenue_df.shape)

print("\nDate Range:")
print("Start Date:", daily_revenue_df["Transaction_Date"].min())
print("End Date:", daily_revenue_df["Transaction_Date"].max())

print("\nDaily Revenue Summary:")
display(
    daily_revenue_df["Daily_Revenue"]
    .describe()
    .round(2)
)

print("\nRevenue Reconciliation:")
print(
    "Transaction-Level Net Revenue:",
    round(cleaned_df["Line_Amount"].sum(), 2)
)
print(
    "Aggregated Daily Net Revenue:",
    round(daily_revenue_df["Daily_Revenue"].sum(), 2)
)

assert np.isclose(
    cleaned_df["Line_Amount"].sum(),
    daily_revenue_df["Daily_Revenue"].sum()
)

assert daily_revenue_df["Transaction_Date"].is_unique
assert daily_revenue_df["Transaction_Date"].is_monotonic_increasing

print("\nFirst Five Daily Records:")
display(daily_revenue_df.head())

Daily Revenue Dataset Shape: (305, 2)

Date Range:
Start Date: 2010-12-01 00:00:00
End Date: 2011-12-09 00:00:00

Daily Revenue Summary:


count       305.00
mean      32109.50
std       17382.67
min        3457.11
25%       20861.90
50%       27871.61
75%       42007.09
max      110806.17
Name: Daily_Revenue, dtype: float64


Revenue Reconciliation:
Transaction-Level Net Revenue: 9793397.24
Aggregated Daily Net Revenue: 9793397.24

First Five Daily Records:


,Transaction_Date,Daily_Revenue
0,2010-12-01,57328.60
1,2010-12-02,46174.28
2,2010-12-03,43510.73
3,2010-12-05,30957.28
4,2010-12-06,50907.56


**Key Observations**
- **The cleaned transaction dataset aggregates into 305 unique daily revenue observations**, covering December 1, 2010, through December 9, 2011, with a total net merchandise revenue of 9,793,397.24.
- **Daily revenue averages 32,109.50, with a median of 27,871.61 and a standard deviation of 17,382.67**, indicating considerable variation in sales performance across observed trading days.
- **The aggregated revenue total matches the cleaned transaction-level revenue exactly**, confirming that daily aggregation preserves the net merchandise revenue definition established during exploratory analysis.

**Data Preparation Decisions**
- **Use `Daily_Revenue` as the forecasting target**, with `Transaction_Date` providing the chronological reference for subsequent modeling.
- **Preserve the 305 observed trading days without introducing artificial zero-revenue records**, pending a separate assessment of the trading calendar.
- **Save the aggregated daily revenue dataset as a reusable Parquet artifact**, allowing subsequent notebooks to load the forecasting target without repeating transaction-level aggregation.

#### Saving the Daily Revenue Dataset

In [8]:
daily_revenue_path = Path("../data/processed/daily_revenue.parquet")

daily_revenue_path.parent.mkdir(parents=True, exist_ok=True)

daily_revenue_df.to_parquet(
    daily_revenue_path,
    index=False
)

saved_daily_revenue = pd.read_parquet(daily_revenue_path)

assert saved_daily_revenue.shape == daily_revenue_df.shape
assert saved_daily_revenue["Transaction_Date"].is_unique
assert saved_daily_revenue["Transaction_Date"].is_monotonic_increasing
assert np.isclose(
    saved_daily_revenue["Daily_Revenue"].sum(),
    cleaned_df["Line_Amount"].sum()
)

print("Daily Revenue Dataset Saved Successfully")
print("File Path:", daily_revenue_path)
print("Saved Dataset Shape:", saved_daily_revenue.shape)
print("Total Net Merchandise Revenue:", round(saved_daily_revenue["Daily_Revenue"].sum(), 2))

Daily Revenue Dataset Saved Successfully
File Path: ..\data\processed\daily_revenue.parquet
Saved Dataset Shape: (305, 2)
Total Net Merchandise Revenue: 9793397.24


#### Examining Trading Calendar Gaps

In [9]:
calendar_dates = pd.date_range(
    start=daily_revenue_df["Transaction_Date"].min(),
    end=daily_revenue_df["Transaction_Date"].max(),
    freq="D"
)

trading_calendar_df = pd.DataFrame({
    "Transaction_Date": calendar_dates
})

trading_calendar_df["Has_Transactions"] = (
    trading_calendar_df["Transaction_Date"].isin(
        daily_revenue_df["Transaction_Date"]
    )
)

trading_calendar_df["Day_of_Week"] = (
    trading_calendar_df["Transaction_Date"].dt.day_name()
)

trading_calendar_df["Is_Saturday"] = (
    trading_calendar_df["Transaction_Date"].dt.dayofweek == 5
)

calendar_summary = pd.DataFrame({
    "Metric": [
        "Total Calendar Days",
        "Observed Trading Days",
        "Days Without Transactions",
        "Missing Saturdays",
        "Missing Non-Saturday Days",
        "Observed Calendar Coverage (%)"
    ],
    "Value": [
        len(trading_calendar_df),
        trading_calendar_df["Has_Transactions"].sum(),
        (~trading_calendar_df["Has_Transactions"]).sum(),
        (
            ~trading_calendar_df["Has_Transactions"]
            & trading_calendar_df["Is_Saturday"]
        ).sum(),
        (
            ~trading_calendar_df["Has_Transactions"]
            & ~trading_calendar_df["Is_Saturday"]
        ).sum(),
        round(
            trading_calendar_df["Has_Transactions"].mean() * 100,
            2
        )
    ]
})

print("Trading Calendar Summary:")
display(calendar_summary)

missing_non_saturday = trading_calendar_df.loc[
    ~trading_calendar_df["Has_Transactions"]
    & ~trading_calendar_df["Is_Saturday"],
    ["Transaction_Date", "Day_of_Week"]
]

print("\nDates Without Recorded Transactions (Excluding Saturdays):")
display(missing_non_saturday.reset_index(drop=True))

print("\nObserved Trading Days by Weekday:")
display(
    trading_calendar_df.groupby("Day_of_Week")["Has_Transactions"]
    .agg(
        Calendar_Days="count",
        Observed_Days="sum"
    )
    .reindex([
        "Monday", "Tuesday", "Wednesday",
        "Thursday", "Friday", "Saturday", "Sunday"
    ])
)

Trading Calendar Summary:


C:\Users\user\AppData\Local\Temp\ipykernel_14280\663922709.py:22: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  trading_calendar_df["Transaction_Date"].dt.dayofweek == 5


,Metric,Value
0,Total Calendar Days,374.00
1,Observed Trading Days,305.00
2,Days Without Transactions,69.00
3,Missing Saturdays,53.00
4,Missing Non-Saturday Days,16.00
5,Observed Calendar Coverage (%),81.55



Dates Without Recorded Transactions (Excluding Saturdays):


,Transaction_Date,Day_of_Week
0,2010-12-24,Friday
1,2010-12-26,Sunday
2,2010-12-27,Monday
3,2010-12-28,Tuesday
4,2010-12-29,Wednesday
5,2010-12-30,Thursday
6,2010-12-31,Friday
7,2011-01-02,Sunday
8,2011-01-03,Monday
9,2011-04-22,Friday



Observed Trading Days by Weekday:


,Calendar_Days,Observed_Days
Day_of_Week,,
Monday,53,47
Tuesday,53,52
Wednesday,54,53
Thursday,54,53
Friday,54,50
Saturday,53,0
Sunday,53,50


**Key Observations**
- **The dataset contains 305 observed trading days across 374 calendar days**, representing 81.55% calendar coverage, with 69 dates having no recorded transactions.
- **All 53 Saturdays lack recorded transactions**, indicating a consistent six-day weekly trading pattern. The remaining 16 missing dates occur between Sunday and Friday.
- **The 16 non-Saturday gaps are concentrated around specific periods**, including a nine-day interruption from December 24, 2010, through January 3, 2011, excluding Saturdays, alongside several shorter gaps during April, May, and August 2011.

**Data Preparation Decisions**
- **Preserve the 305 observed revenue values without replacing missing dates with zero**, since the absence of transaction records does not independently confirm zero sales.
- **Maintain a separate calendar reference identifying observed and unobserved dates**, allowing future feature engineering to distinguish trading-day patterns from calendar-day intervals.
- **Use an observed-trading-day forecasting framework initially**, while explicitly accounting for irregular calendar gaps and evaluating the implications for forecast horizons and historical lag features.

#### Saving the Trading Calendar Reference

In [10]:
calendar_reference_df = trading_calendar_df[
    [
        "Transaction_Date",
        "Day_of_Week",
        "Is_Saturday",
        "Has_Transactions"
    ]
].copy()

calendar_reference_df["Calendar_Status"] = np.select(
    [
        calendar_reference_df["Has_Transactions"],
        calendar_reference_df["Is_Saturday"]
    ],
    [
        "Observed Trading Day",
        "Unobserved Saturday"
    ],
    default="Unobserved Non-Saturday"
)

calendar_reference_path = Path(
    "../data/processed/trading_calendar.parquet"
)

calendar_reference_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

calendar_reference_df.to_parquet(
    calendar_reference_path,
    index=False
)

saved_calendar_df = pd.read_parquet(
    calendar_reference_path
)

assert len(saved_calendar_df) == 374
assert saved_calendar_df["Transaction_Date"].is_unique
assert saved_calendar_df["Has_Transactions"].sum() == 305
assert (
    saved_calendar_df["Calendar_Status"]
    == "Unobserved Saturday"
).sum() == 53

print("Trading Calendar Saved Successfully")
print("File Path:", calendar_reference_path)
print("Saved Dataset Shape:", saved_calendar_df.shape)

print("\nCalendar Status Summary:")
display(
    saved_calendar_df["Calendar_Status"]
    .value_counts()
    .rename_axis("Calendar_Status")
    .reset_index(name="Day_Count")
)

print("\nFirst Five Records:")
display(saved_calendar_df.head())

Trading Calendar Saved Successfully
File Path: ..\data\processed\trading_calendar.parquet
Saved Dataset Shape: (374, 5)

Calendar Status Summary:


,Calendar_Status,Day_Count
0,Observed Trading Day,305
1,Unobserved Saturday,53
2,Unobserved Non-Saturday,16



First Five Records:


,Transaction_Date,Day_of_Week,Is_Saturday,Has_Transactions,Calendar_Status
0,2010-12-01,Wednesday,False,True,Observed Trading Day
1,2010-12-02,Thursday,False,True,Observed Trading Day
2,2010-12-03,Friday,False,True,Observed Trading Day
3,2010-12-04,Saturday,True,False,Unobserved Saturday
4,2010-12-05,Sunday,False,True,Observed Trading Day


#### Preparing Chronological Train and Test Splits

In [11]:
daily_revenue_df = daily_revenue_df.sort_values(
    "Transaction_Date"
).reset_index(drop=True)

train_size = int(len(daily_revenue_df) * 0.80)

train_df = daily_revenue_df.iloc[:train_size].copy()
test_df = daily_revenue_df.iloc[train_size:].copy()

assert len(train_df) + len(test_df) == len(daily_revenue_df)
assert train_df["Transaction_Date"].max() < test_df["Transaction_Date"].min()
assert set(train_df["Transaction_Date"]).isdisjoint(
    set(test_df["Transaction_Date"])
)

split_summary = pd.DataFrame({
    "Dataset": ["Training", "Testing", "Complete"],
    "Observations": [
        len(train_df),
        len(test_df),
        len(daily_revenue_df)
    ],
    "Percentage": [
        len(train_df) / len(daily_revenue_df) * 100,
        len(test_df) / len(daily_revenue_df) * 100,
        100
    ],
    "Start_Date": [
        train_df["Transaction_Date"].min(),
        test_df["Transaction_Date"].min(),
        daily_revenue_df["Transaction_Date"].min()
    ],
    "End_Date": [
        train_df["Transaction_Date"].max(),
        test_df["Transaction_Date"].max(),
        daily_revenue_df["Transaction_Date"].max()
    ]
})

print("Chronological Split Summary:")
display(split_summary.round(2))

print("\nTraining Revenue Summary:")
display(train_df["Daily_Revenue"].describe().round(2))

print("\nTesting Revenue Summary:")
display(test_df["Daily_Revenue"].describe().round(2))

Chronological Split Summary:


C:\Users\user\AppData\Local\Temp\ipykernel_14280\1622255796.py:41: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(split_summary.round(2))


,Dataset,Observations,Percentage,Start_Date,End_Date
0,Training,244,80.0,2010-12-01,2011-09-29
1,Testing,61,20.0,2011-09-30,2011-12-09
2,Complete,305,100.0,2010-12-01,2011-12-09



Training Revenue Summary:


count       244.00
mean      27925.42
std       14347.88
min        3457.11
25%       19521.46
50%       25412.92
75%       33576.83
max      108335.17
Name: Daily_Revenue, dtype: float64


Testing Revenue Summary:


count        61.00
mean      48845.82
std       18472.27
min       11623.58
25%       36091.45
50%       47166.66
75%       60423.71
max      110806.17
Name: Daily_Revenue, dtype: float64

**Key Observations**
- **The dataset is divided chronologically into 244 training observations (80%) and 61 testing observations (20%)**, with the training period ending September 29, 2011, and the testing period beginning September 30, 2011.
- **Average daily revenue increases from 27,925.42 in training to 48,845.82 in testing**, representing approximately 74.92% higher average revenue in the holdout period.
- **The testing period exhibits greater absolute revenue variability**, with a standard deviation of 18,472.27 compared with 14,347.88 during training. The maximum daily revenue also increases from 108,335.17 to 110,806.17.

**Data Preparation Decisions**
- **Preserve the chronological 80/20 split without shuffling**, ensuring that forecasting models are evaluated on observations occurring after the training period.
- **Keep the testing period isolated from model selection and hyperparameter tuning**, using time-series cross-validation exclusively within the training data.
- **Account for differences in revenue levels between training and testing**, recognizing that the holdout period presents a challenging evaluation scenario involving higher sales activity.

#### Saving Chronological Train and Test Datasets

In [12]:
train_path = Path("../data/processed/daily_revenue_train.parquet")
test_path = Path("../data/processed/daily_revenue_test.parquet")

train_df.to_parquet(train_path, index=False)
test_df.to_parquet(test_path, index=False)

saved_train_df = pd.read_parquet(train_path)
saved_test_df = pd.read_parquet(test_path)

assert len(saved_train_df) == 244
assert len(saved_test_df) == 61
assert (
    saved_train_df["Transaction_Date"].max()
    < saved_test_df["Transaction_Date"].min()
)
assert np.isclose(
    saved_train_df["Daily_Revenue"].sum()
    + saved_test_df["Daily_Revenue"].sum(),
    daily_revenue_df["Daily_Revenue"].sum()
)

print("Chronological Datasets Saved Successfully")
print("\nTraining Dataset:", train_path)
print("Training Shape:", saved_train_df.shape)
print("\nTesting Dataset:", test_path)
print("Testing Shape:", saved_test_df.shape)

print("\nRevenue Reconciliation:")
print(
    "Combined Train/Test Revenue:",
    round(
        saved_train_df["Daily_Revenue"].sum()
        + saved_test_df["Daily_Revenue"].sum(),
        2
    )
)
print(
    "Original Daily Revenue:",
    round(daily_revenue_df["Daily_Revenue"].sum(), 2)
)

Chronological Datasets Saved Successfully

Training Dataset: ..\data\processed\daily_revenue_train.parquet
Training Shape: (244, 2)

Testing Dataset: ..\data\processed\daily_revenue_test.parquet
Testing Shape: (61, 2)

Revenue Reconciliation:
Combined Train/Test Revenue: 9793397.24
Original Daily Revenue: 9793397.24


#### Summarizing Data Preparation

The data preparation process transformed the original Online Retail transaction dataset into a structured daily revenue series suitable for time-series forecasting.

**Key Outcomes**
- **Cleaned Transaction Data:** Retained 538,998 merchandise invoice lines after excluding 2,911 non-merchandise records, preserving cancellations, negative revenue contributions, and duplicate invoice lines.
- **Daily Revenue Aggregation:** Generated 305 chronological revenue observations, with total net merchandise revenue of 9,793,397.24.
- **Trading Calendar:** Identified 69 dates without recorded transactions, including 53 Saturdays and 16 other dates, without introducing artificial zero-revenue observations.
- **Chronological Splitting:** Allocated 244 observations to training and 61 to testing, maintaining temporal order and preserving the complete revenue total.
- **Data Reusability:** Saved cleaned transactions, daily revenue, trading-calendar information, and chronological train/test datasets as reusable Parquet artifacts.

**Forecasting Considerations**
- **Revenue Distribution Shift:** Average daily revenue is approximately 74.92% higher in the testing period, creating a challenging forecasting evaluation scenario.
- **Temporal Structure:** Historical revenue exhibits changing levels, recurring trading-day patterns, and irregular calendar gaps that require careful treatment during feature engineering.
- **Data Leakage Prevention:** Future lagged and rolling features must use only information available at each forecast origin, while the testing period remains excluded from model selection.
- **Forecast Horizon:** The initial forecasting framework targets the next seven observed trading days, with calendar-date interpretation and multi-step evaluation requiring explicit treatment.

**Final Outcome**

The prepared datasets establish a reproducible foundation for developing and evaluating daily retail revenue forecasting models while preserving the chronological structure and net merchandise revenue definition established during exploratory analysis.